### **Stage 2: LangGraph (multi_agent)**

##### **Importing Libraries**

In [17]:
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command
from langchain_core.messages import SystemMessage, HumanMessage

##### **Model**

In [18]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

api_key = os.getenv("telecom_assg2")

if not api_key:
    raise RuntimeError("Set telecom_assg2 in your .env file before running the agent cells.")

model_name = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")

model = ChatGroq(model= model_name, temperature= 0, api_key= api_key)

##### **Mock Telecom Database**

In [19]:
customer_details = {
    "AC10234": {"name": "L. Kim",       "phone_number": "212-555-0142", "plan": "unlimited_plus", "area_code": "212", "monthly_bill_usd": 85,  "data_used_gb": 42},
    "AC20458": {"name": "D. Osei",      "phone_number": "310-555-0198", "plan": "standard_20gb",  "area_code": "310", "monthly_bill_usd": 45,  "data_used_gb": 12},
    "AC30671": {"name": "P. Novak",     "phone_number": "404-555-0113", "plan": "basic_5gb",      "area_code": "404", "monthly_bill_usd": 30,  "data_used_gb": 4.5},
    "AC40892": {"name": "R. Alvarez",   "phone_number": "512-555-0176", "plan": "business_50gb",  "area_code": "512", "monthly_bill_usd": 120, "data_used_gb": 38},
    "AC50103": {"name": "S. Chen",      "phone_number": "606-555-0159", "plan": "standard_20gb",  "area_code": "606", "monthly_bill_usd": 45,  "data_used_gb": 15},
    "AC60217": {"name": "T. Barros",    "phone_number": "213-555-0134", "plan": "basic_5gb",      "area_code": "213", "monthly_bill_usd": 30,  "data_used_gb": 6.2},
    "AC70345": {"name": "N. Whitfield", "phone_number": "718-555-0187", "plan": "unlimited_plus", "area_code": "718", "monthly_bill_usd": 85,  "data_used_gb": 55},
    "AC80456": {"name": "M. Kowalski",  "phone_number": "702-555-0121", "plan": "business_50gb",  "area_code": "702", "monthly_bill_usd": 120, "data_used_gb": 21}
}

plan_details = {
    "basic_5gb":      {"data_gb": 5,           "minutes": "unlimited", "price_usd": 30,  "overage_fee_per_gb": 10},
    "standard_20gb":  {"data_gb": 20,          "minutes": "unlimited", "price_usd": 45,  "overage_fee_per_gb": 8},
    "unlimited_plus": {"data_gb": "unlimited", "minutes": "unlimited", "price_usd": 85,  "overage_fee_per_gb": 0},
    "business_50gb":  {"data_gb": 50,          "minutes": "unlimited", "price_usd": 120, "overage_fee_per_gb": 6},
    "family_100gb":   {"data_gb": 100,         "minutes": "unlimited", "price_usd": 150, "overage_fee_per_gb": 5}
}

network_status = {
    "212": {"status": "Outage",      "outage_hours": 18, "affected_services": ["voice", "data"],       "region": "New York, NY",    "technician_dispatched": True},
    "310": {"status": "Operational", "outage_hours": 0,  "affected_services": [],                      "region": "Los Angeles, CA", "technician_dispatched": False},
    "404": {"status": "Degraded",    "outage_hours": 3,  "affected_services": ["data"],                "region": "Atlanta, GA",     "technician_dispatched": False},
    "512": {"status": "Operational", "outage_hours": 0,  "affected_services": [],                      "region": "Austin, TX",      "technician_dispatched": False},
    "606": {"status": "Outage",      "outage_hours": 30, "affected_services": ["voice", "data", "sms"],"region": "Lexington, KY",   "technician_dispatched": True},
    "213": {"status": "Operational", "outage_hours": 0,  "affected_services": [],                      "region": "Los Angeles, CA", "technician_dispatched": False},
    "718": {"status": "Degraded",    "outage_hours": 5,  "affected_services": ["voice"],               "region": "Brooklyn, NY",    "technician_dispatched": True},
    "702": {"status": "Operational", "outage_hours": 0,  "affected_services": [],                      "region": "Las Vegas, NV",   "technician_dispatched": False}
}

##### **Defining Tools**

In [20]:
# TOOL 1
def lookup_account(customer_id: str):
    """Look up an account by its account_id, e.g., 'AC10234'"""
    
    customer_info = customer_details.get(customer_id.upper())
    
    # If customer information not found, return an error message
    if not customer_info:
        return {"error": f"No information found for {customer_id}."}    
    return customer_info
    
# TOOL 2
def check_network_status(area_code: str | int):
    """Check the current network status for a 3-digit area code, e.g. '212'."""
    
    area_code = str(area_code).strip()

    # Look up the network status.
    status = network_status.get(area_code)
    
    # If area code is not recognized, return an error message
    if not status:
        return {"error": f"No network status found for area code '{area_code}"}
    return status    

# TOOL 3
def request_plan_change(customer_id: str, new_plan: str):
    """Check whether a requested plan exists and calculate the monthly price difference. 
    Does NOT change the plan — only quotes it."""

    account = lookup_account(customer_id)
    
    # Check whether the customer exists
    if "error" in account:
        return {"error": f"Unknown customer ID '{customer_id}'"}
    
    new_plan = new_plan.lower()
    target = plan_details.get(new_plan)
    
    # Check whether the requested plan exists, If target is None, the requested plan does not exist.
    if not target:
        return {"available": False, "reason": f"Unknown plan '{new_plan}'. Valid: basic_5gb, standard_20gb, unlimited_plus, business_50gb."}
    
    current = plan_details[account["plan"]]
    
    # Calculate the price difference
    price_diff = round(target["price_usd"] - current["price_usd"], 2)
    
    # Return the plan-change quotation
    return {"available": True, "current_plan": account["plan"], "new_plan": new_plan, "price_diff_usd": price_diff}
  

##### **Creating Nodes**

In [ ]:
# State
class SupportState(TypedDict):
    messages: Annotated[list, add_messages]
    next: str
    handled: list[str]

# Supervisor Instructions
supervisor_system = """
- You are the TeleAssist support supervisor. Given the conversation and which specialists have already run, decide the single next step. 
- Reply with exactly one word: 'account', 'network_status', 'plan_change' or 'finish'. 
- Only pick a specialist that hasn't already run for this request unless the customer explicitly asks again. 
- Pick 'finish' once every part of the customer's request has been addressed."""

def get_user_message(state: SupportState):

    for message in reversed(state["messages"]):
        if isinstance(message, HumanMessage):
            return message.content

        if isinstance(message, dict):
            if message.get("role") == "user":
                return message.get("content", "")
    return "" 

# Node 1
def account_node(state: SupportState):

    # Get the original request from the user.
    last_user = get_user_message(state)

    # This will store the customer's account ID and search through all customer IDs in database.
    account_id = None
    for account_code in customer_details:
        if account_code.lower() in last_user.lower():
            account_id = account_code
            break
    
    # If no account ID was found, ask the user for right one.
    if account_id is None:
        text = "Please provide a valid customer ID."
    else:
        account = lookup_account(account_id)

        # If the lookup tool returned an error, show that error.
        if "error" in account:
            text = account["error"]
        else:
            text = (
                f"Account {account_id}: "
                f"Customer: {account['name']}, "
                f"Plan: {account['plan']}, "
                f"Monthly bill: ${account['monthly_bill_usd']}, "
                f"Data used: {account['data_used_gb']} GB, "
                f"Area code: {account['area_code']}."
            )

    return {"messages": [{"role": "assistant", "content": text}],
            "handled": state.get("handled", []) + ["account"]}
# Node 2
def network_status_node(state: SupportState):

    # Get the original user request
    last_user = get_user_message(state)

    # Find the account ID directly from the customer database
    account_id = next((
            account_code
            for account_code in customer_details
            if account_code.lower() in last_user.lower()), None)

    # If no customer ID was found, return an error message.
    if account_id is None:
        text = "Please provide a valid customer ID."
    else:
        account = lookup_account(account_id)

        if "error" in account:
            text = account["error"]
        else:
            area_code = account["area_code"]
            status = check_network_status(area_code)

            # Check whether the network lookup returned an error.
            if "error" in status:
                text = status["error"]
            else:
                affected = status["affected_services"]
                affected_text = (", ".join(affected) if affected else "none")

                text = (
                    f"Network status for account {account_id}: "
                    f"{status['status']}. "
                    f"Region: {status['region']}. "
                    f"Outage duration: {status['outage_hours']} hours. "
                    f"Affected services: {affected_text}. "
                    f"Technician dispatched: "
                    f"{status['technician_dispatched']}."
                )

    return {"messages": [{"role": "assistant", "content": text}],
            "handled": state.get("handled", []) + ["network_status"]}

# Node 3
def plan_change_node(state: SupportState):

    # Get the original user request.
    last_user = get_user_message(state)

    # Find the customer's account ID
    account_id = None
    for account_code in customer_details:
        if account_code.lower() in last_user.lower():
            account_id = account_code
            break
    
    # Find the requested plan
    requested_plan = None
    for plan_name in plan_details:
        if plan_name.lower() in last_user.lower():
            requested_plan = plan_name
            break

    # Validate account ID
    if account_id is None:
        text = "Please provide a valid customer ID."
        return {"messages": [{"role": "assistant", "content": text}],
                "handled": state.get("handled", []) + ["plan_change"]}

    # Validate requested plan
    if requested_plan is None:
        text = "Please provide a valid plan name."
        return {"messages": [{"role": "assistant", "content": text}],
                "handled": state.get("handled", []) + ["plan_change"]}

    # Calculate the plan-change quote
    quote = request_plan_change(account_id, requested_plan)

    # Human in the loop approval
    approval = interrupt({
        "action": "plan_change",
        "customer_id": account_id,
        "requested_plan": requested_plan,
        "calculated_quote": quote,
        "question": "Approve this plan change? (yes/no)"
    })

    # Approval Process
    if str(approval).strip().lower() in ("yes", "y", "approve", "approved"):

        if quote.get("available"):
            difference = quote["price_diff_usd"]

            text = (
                f"Plan change approved. Changing from {quote['current_plan']} to {quote['new_plan']} changes the monthly bill by ${difference:+.2f}. "
                f"The current tool only provides the quote and does not modify the account."
            )
        else:
            text = "The requested plan is not available."
    else:
        text = "The plan change was not approved."

    return {"messages": [{"role": "assistant", "content": text}],
            "handled": state.get("handled", []) + ["plan_change"]}
      
# These rules tell the supervisor which type of request corresponds to which specialist node.
supervisor_rules = [
    ("network_status", ("network", "outage", "connectivity")),
    ("account", ("account", "customer")),
    ("plan_change", ("change my plan", "change plan", "switch my plan",
                          "switch plan", "upgrade my plan", "downgrade my plan"))
]

# Supervisor Node
def supervisor_node(state: SupportState):

    handled = state.get("handled", [])
    user_message = get_user_message(state).lower()

    # Check each supervisor rule.
    for tag, keywords in supervisor_rules:
            if tag not in handled and any(k in user_message for k in keywords):
                return {"next": tag}
    return {"next": "finish"}

# Router
def route_from_supervisor(state: SupportState):
    return {
        "account": "account_node",
        "network_status": "network_status_node",
        "plan_change": "plan_change_node",
        "finish": END
    }[state["next"]]

##### **Graph**

In [22]:
builder = StateGraph(SupportState)

# Add nodes to the graph
builder.add_node("supervisor", supervisor_node)
builder.add_node("account_node", account_node)
builder.add_node("network_status_node", network_status_node)
builder.add_node("plan_change_node", plan_change_node)

# Add edges to the graph
# Start Edge
builder.add_edge(START, "supervisor")

# Conditional Edge
builder.add_conditional_edges("supervisor", route_from_supervisor)

# End Edges 
builder.add_edge("account_node", "supervisor")
builder.add_edge("network_status_node", "supervisor")
builder.add_edge("plan_change_node", "supervisor")

checkpointer = InMemorySaver()
stage2_graph = builder.compile(checkpointer= checkpointer)

In [23]:
# Create a thread id
config = {"configurable": {"thread_id": "customer-001"}}

result = stage2_graph.invoke(
    {"messages": [{"role": "user", "content": ("Check network status for account AC80456. "
                                               "Check my account, tell me if there is a network outage" 
                                               "and I want to change my plan to business_50gb.")}],
     "handled": []},
    config= config)

# Checks whether the graph paused
print("Paused at interrupt:", result.get("__interrupt__"))

print("--------------------------------------------------------")

# Resume the paused graph
final_state = stage2_graph.invoke(Command(resume="yes"), config= config)

print("Final Answer:")

for message in final_state["messages"]:
    if isinstance(message, dict):
        role = message.get("role")
        content = message.get("content")
    else:
        role = message.type
        content = message.content

    print(f"[{role}] {content}")

Paused at interrupt: [Interrupt(value={'action': 'plan_change', 'customer_id': 'AC80456', 'requested_plan': 'business_50gb', 'calculated_quote': {'available': True, 'current_plan': 'business_50gb', 'new_plan': 'business_50gb', 'price_diff_usd': 0}, 'question': 'Approve this plan change? (yes/no)'}, id='0cd30a2d957c829346ca93727ea41d59')]
--------------------------------------------------------
Final Answer:
[human] Check network status for account AC80456. Check my account, tell me if there is a network outageand I want to change my plan to business_50gb.
[ai] Network status for account AC80456: Operational. Region: Las Vegas, NV. Outage duration: 0 hours. Affected services: none. Technician dispatched: False.
[ai] Account AC80456: Customer: M. Kowalski, Plan: business_50gb, Monthly bill: $120, Data used: 21 GB, Area code: 702.
[ai] Plan change approved. Changing from business_50gb to business_50gb changes the monthly bill by $+0.00. The current tool only provides the quote and does no